In [ ]:
import torch.nn.functional as F_func
import math
import numpy as np
import torch
torch.set_printoptions(sci_mode=False)
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
from typing import List
import random
import lightning as L
from tqdm import tqdm 

import articulate as art
from config import *
from utils import *
from helpers import *
from typing import Sequence

from utils.model_utils import reduced_pose_to_full
from articulate.math import r6d_to_rotation_matrix
from articulate.evaluator import MeanPerJointErrorEvaluator
from articulate.math import RotationRepresentation

In [ ]:

class PoseDataset(Dataset):
    def __init__(self, fold: str='train', evaluate: str=None, finetune: str=None):
        super().__init__()
        self.fold = fold
        self.evaluate = evaluate
        self.finetune = finetune
        self.bodymodel = art.model.ParametricModel(paths.smpl_file)
        self.combos = list(amass.combos.items())
        self.data = self._prepare_dataset()

    def _get_data_files(self, data_folder):
        if self.fold == 'train':
            return self._get_train_files(data_folder)
        elif self.fold == 'test':
            return self._get_test_files()
        else:
            raise ValueError(f"Unknown data fold: {self.fold}.")

    def _get_train_files(self, data_folder):
        if self.finetune:
            return [datasets.finetune_datasets[self.finetune]]
        else:
            return [x.name for x in data_folder.iterdir() if not x.is_dir()]

    def _get_test_files(self):
        return [datasets.test_datasets[self.evaluate]]

    def _prepare_dataset(self):
        data_folder = paths.processed_datasets / ('eval' if (self.finetune or self.evaluate) else '')
        data_files = self._get_data_files(data_folder)

        #print(f"\n{'='*60}")
        #print(f"Loading datasets for {self.fold.upper()} mode")
        #print(f"Datasets: {data_files}")
        #print(f"{'='*60}\n")

        data = {key: [] for key in [
            'imu_inputs', 'pose_outputs', 'joint_outputs',
            'tran_outputs', 'vel_outputs', 'foot_outputs'
        ]}

        for data_file in tqdm(data_files):
            try:
                file_data = torch.load(data_folder / data_file)
                self._process_file_data(file_data, data)
            except Exception as e:
                print(f"Error processing {data_file}: {e}")

        #print("\nTotal pose windows stored:", len(data['pose_outputs']))
        return data

    def _process_file_data(self, file_data, data):
        accs, oris, poses, trans = file_data['acc'], file_data['ori'], file_data['pose'], file_data['tran']
        joints = file_data.get('joint', [None] * len(poses))
        foots = file_data.get('contact', [None] * len(poses))

        for acc, ori, pose, tran, joint, foot in zip(accs, oris, poses, trans, joints, foots):


            acc, ori = acc[:, :5]/amass.acc_scale, ori[:, :5]

            pose_global, joint = self.bodymodel.forward_kinematics(
                pose=pose.view(-1, 216)
            )

            pose = pose if self.evaluate else pose_global.view(-1, 24, 3, 3)
            joint = joint.view(-1, 24, 3)

            #print("Pose after forward kinematics:", pose.shape)

            self._process_combo_data(acc, ori, pose, joint, tran, foot, data)

    def _process_combo_data(self, acc, ori, pose, joint, tran, foot, data):

        for combo_name, c in self.combos:

            #print("\n" + "="*50)
            #print("Processing combo:", combo_name)

            combo_acc = torch.zeros_like(acc)
            combo_ori = torch.zeros_like(ori)
            combo_acc[:, c] = acc[:, c]
            combo_ori[:, c] = ori[:, c]

            imu_input = torch.cat([combo_acc.flatten(1), combo_ori.flatten(1)], dim=1)

            data_len = len(imu_input) if self.evaluate else datasets.window_length

            for key, value in zip(
                ['imu_inputs', 'pose_outputs', 'joint_outputs', 'tran_outputs'],
                [imu_input, pose, joint, tran]
            ):
                # data[key].extend(torch.split(value, data_len))
                splits = torch.split(value, data_len)

                # remove last if smaller than full window
                if splits[-1].shape[0] < data_len:
                    splits = splits[:-1]

                data[key].extend(splits)

            if not (self.evaluate or self.finetune):
                self._process_translation_data(joint, tran, foot, data_len, data)

    def _process_translation_data(self, joint, tran, foot, data_len, data):


        root_vel = torch.cat((torch.zeros(1, 3), tran[1:] - tran[:-1]))
        vel = torch.cat((torch.zeros(1, 24, 3), torch.diff(joint, dim=0)))
        vel[:, 0] = root_vel

        vel = vel * (datasets.fps / amass.vel_scale)
        vel_splits = torch.split(vel, data_len)

        data['vel_outputs'].extend(vel_splits)
        data['foot_outputs'].extend(torch.split(foot, data_len))

    def __getitem__(self, idx):

        imu = self.data['imu_inputs'][idx].float()
        joint = self.data['joint_outputs'][idx].float()
        tran = self.data['tran_outputs'][idx].float()

        num_pred_joints = len(amass.pred_joints_set)

        pose = art.math.rotation_matrix_to_r6d(
            self.data['pose_outputs'][idx]
        ).reshape(-1, num_pred_joints, 6)[:, amass.pred_joints_set] \
         .reshape(-1, 6*num_pred_joints)

        if self.evaluate or self.finetune:
            return imu, pose, joint, tran

        vel = self.data['vel_outputs'][idx].float()
        contact = self.data['foot_outputs'][idx].float()

        return imu, pose, joint, tran, vel, contact

    def __len__(self):
        return len(self.data['imu_inputs'])
    




def pad_seq(batch):
    """Pad sequences to same length for RNN."""
    def _pad(sequence):
        padded = nn.utils.rnn.pad_sequence(sequence, batch_first=True)
        lengths = [seq.shape[0] for seq in sequence]
        return padded, lengths

    inputs, poses, joints, trans = zip(*[(item[0], item[1], item[2], item[3]) for item in batch])
    inputs, input_lengths = _pad(inputs)
    poses, pose_lengths = _pad(poses)
    joints, joint_lengths = _pad(joints)
    trans, tran_lengths = _pad(trans)
    
    outputs = {'poses': poses, 'joints': joints, 'trans': trans}
    output_lengths = {'poses': pose_lengths, 'joints': joint_lengths, 'trans': tran_lengths}

    if len(batch[0]) > 5: # include velocity and foot contact, if available
        vels, foots = zip(*[(item[4], item[5]) for item in batch])

        # foot contact 
        foot_contacts, foot_contact_lengths = _pad(foots)
        outputs['foot_contacts'], output_lengths['foot_contacts'] = foot_contacts, foot_contact_lengths

        # root velocities
        vels, vel_lengths = _pad(vels)
        outputs['vels'], output_lengths['vels'] = vels, vel_lengths

    return (inputs, input_lengths), (outputs, output_lengths)


class PoseDataModule(L.LightningDataModule):
    def __init__(self, finetune: str = None):
        super().__init__()
        self.finetune = finetune
        self.hypers = finetune_hypers if self.finetune else train_hypers

    def setup(self, stage: str):
        if stage == 'fit':
            dataset = PoseDataset(fold='train', finetune=self.finetune)
            train_size = int(0.9 * len(dataset))
            val_size = len(dataset) - train_size
            self.train_dataset, self.val_dataset = random_split(dataset, [train_size, val_size])
        elif stage == 'test':
            self.test_dataset = PoseDataset(fold='test', finetune=self.finetune)

    def _dataloader(self, dataset):
        return DataLoader(
            dataset, 
            batch_size=self.hypers.batch_size, 
            collate_fn=pad_seq, 
            num_workers=0, #self.hypers.num_workers, 
            shuffle=True, 
            drop_last=True
        )

    def train_dataloader(self):
        return self._dataloader(self.train_dataset)

    def val_dataloader(self):
        return self._dataloader(self.val_dataset)

    def test_dataloader(self):
        return self._dataloader(self.test_dataset)

In [ ]:

# ---------------- Diffusion hyperparameters ----------------
TIMESTEPS = 1000
BETA_START = 1e-4
BETA_END = 0.02

betas = torch.linspace(BETA_START, BETA_END, TIMESTEPS)
alphas = 1.0 - betas
alpha_bar = torch.cumprod(alphas, dim=0)


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Add positional encoding to input.

        x: (B, T, D)
        """
        T = x.size(1)
        return x + self.pe[:T].unsqueeze(0)


class TemporalTransformerDenoiser(nn.Module):
    def __init__(
        self,
        feature_dim: int,
        d_model: int = 256,
        nhead: int = 8,
        num_layers: int = 4,
        dim_feedforward: int = 512,
        dropout: float = 0.1,
    ):
        """Temporal Transformer denoiser.

        Inputs / outputs: (B, T, feature_dim)
        """
        super().__init__()

        self.in_proj = nn.Linear(feature_dim, d_model)
        self.pos_enc = SinusoidalPositionalEncoding(d_model)

        # simple time embedding to condition on diffusion timestep
        self.time_mlp = nn.Sequential(
            nn.Linear(1, d_model),
            nn.SiLU(),
            nn.Linear(d_model, d_model),
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.out_proj = nn.Linear(d_model, feature_dim)
    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """Predict diffusion noise.

        x: (B, T, F)  -> noisy pose x_t
        t: (B,)       -> integer timesteps in [0, TIMESTEPS)
        """
        B = x.size(0)

        h = self.in_proj(x)          # (B, T, d_model)
        h = self.pos_enc(h)

        # normalize timestep to [0, 1] and embed
        t_norm = t.float().unsqueeze(1) / max(TIMESTEPS - 1, 1)
        t_emb = self.time_mlp(t_norm).unsqueeze(1)  # (B, 1, d_model)

        # broadcast time embedding over sequence length
        h = h + t_emb

        h = self.encoder(h)  # (B, T, d_model)
        return self.out_proj(h)  # (B, T, F), predicted noise
    

In [ ]:

datamodule = PoseDataModule(finetune=None)
datamodule.setup(stage='fit')

train_loader = datamodule.train_dataloader()
val_loader   = datamodule.val_dataloader()

device = "cuda" if torch.cuda.is_available() else "cpu"

model = TemporalTransformerDenoiser(
    feature_dim=144,
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=256,
    dropout=0.1,
).to(device)

In [ ]:
from articulate.math import r6d_to_rotation_matrix
from viewers.smpl_viewer import SMPLViewer

In [ ]:
# torch.save({
#         'denoiser': model.state_dict(),
#         'imu_encoder': imu_encoder.state_dict(),
#     }, "pose_diffusion_transformer.pth")
#     #print("Model saved.")


In [ ]:
model.load_state_dict(torch.load(r"C:\deepika\2_mobileposer\MobilePoser\mobileposer\codes_\pose_diffusion_transformer.pth"))  # Or your saved model path


In [ ]:


model.eval()

k = 0

for i, batch in enumerate(train_loader):
    if i == k:
        break

(inputs, input_lengths), (outputs, output_lengths) = batch

x = outputs["poses"].to(device)
trans = outputs["trans"].to(device)
imu = inputs.to(device)

B, T, Fdim = x.shape
lengths = output_lengths["poses"]

_betas    = betas.to(device)
_alphas   = alphas.to(device)
_alpha_bar = alpha_bar.to(device)

x_t = torch.randn_like(x)                         # start from pure noise

with torch.no_grad():
    for t_val in reversed(range(TIMESTEPS)):
        t_tensor = torch.full((B,), t_val, device=device, dtype=torch.long)

        noise_pred = model(x_t, t_tensor)          # predicted noise

        beta_t     = _betas[t_val]
        alpha_t    = _alphas[t_val]
        alpha_bar_t = _alpha_bar[t_val]

        # DDPM mean: mu_theta(x_t, t)
        coeff = beta_t / torch.sqrt(1.0 - alpha_bar_t)
        mean  = (1.0 / torch.sqrt(alpha_t)) * (x_t - coeff * noise_pred)

        if t_val > 0:
            sigma = torch.sqrt(beta_t)
            x_t = mean + sigma * torch.randn_like(x_t)
        else:
            x_t = mean                              # no noise at t = 0

pred = x_t                                          # final denoised prediction

gt_batch = x
pred_batch = pred
tran_batch = trans
lengths = output_lengths["poses"]

batch_size = 2

for b in range(batch_size):
    L = int(lengths[b])

    gt_pose_r6d   = gt_batch[b, :L]
    pred_pose_r6d = pred_batch[b, :L]
    tran_gt       = tran_batch[b, :L]
    tran_pred     = torch.zeros_like(tran_gt)  # dummy translation predictions for visualization

    gt_pose_rot = r6d_to_rotation_matrix(
        gt_pose_r6d.view(-1, 24, 6)
    ).view(-1, 24, 3, 3)

    pred_pose_rot = r6d_to_rotation_matrix(
        pred_pose_r6d.view(-1, 24, 6)
    ).view(-1, 24, 3, 3)

    # --- Ground Truth video (shown alone) ---
    #print(f"[Seq {b+1}/{batch_size}, len={L}] Ground Truth")
    viewer_gt = SMPLViewer(fps=25)
    # pass gt as pose_p so it renders without needing GT env var
    dummy = torch.zeros_like(gt_pose_rot)
    # viewer_gt.view(gt_pose_rot, tran_gt, dummy, tran_gt, with_tran=True)

    # --- Prediction video (shown alone) ---
    #print(f"[Seq {b+1}/{batch_size}, len={L}] Prediction")
    viewer_pred = SMPLViewer(fps=25)
    viewer_pred.view(pred_pose_rot, tran_gt, dummy, tran_pred, with_tran=False)


# transformer


In [ ]:
TIMESTEPS = 1000
BETA_START = 1e-4
BETA_END = 0.02

betas = torch.linspace(BETA_START, BETA_END, TIMESTEPS)
alphas = 1.0 - betas
alpha_bar = torch.cumprod(alphas, dim=0)


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Add positional encoding to input.

        x: (B, T, D)
        """
        T = x.size(1)
        return x + self.pe[:T].unsqueeze(0)


class TemporalTransformerDenoiser(nn.Module):
    def __init__(
        self,
        feature_dim: int,
        d_model: int = 256,
        nhead: int = 8,
        num_layers: int = 4,
        dim_feedforward: int = 512,
        dropout: float = 0.1,
    ):
        """Temporal Transformer denoiser.

        Inputs / outputs: (B, T, feature_dim)
        """
        super().__init__()

        self.in_proj = nn.Linear(feature_dim, d_model)
        self.pos_enc = SinusoidalPositionalEncoding(d_model)

        # simple time embedding to condition on diffusion timestep
        self.time_mlp = nn.Sequential(
            nn.Linear(1, d_model),
            nn.SiLU(),
            nn.Linear(d_model, d_model),
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.out_proj = nn.Linear(d_model, feature_dim)
    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """Predict diffusion noise.

        x: (B, T, F)  -> noisy pose x_t
        t: (B,)       -> integer timesteps in [0, TIMESTEPS)
        """
        B = x.size(0)

        h = self.in_proj(x)          # (B, T, d_model)
        h = self.pos_enc(h)

        # normalize timestep to [0, 1] and embed
        t_norm = t.float().unsqueeze(1) / max(TIMESTEPS - 1, 1)
        t_emb = self.time_mlp(t_norm).unsqueeze(1)  # (B, 1, d_model)

        # broadcast time embedding over sequence length
        h = h + t_emb

        h = self.encoder(h)  # (B, T, d_model)
        return self.out_proj(h)  # (B, T, F), predicted noise
    


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Use temporal Transformer denoiser (same input/output shape as before)
# model = TemporalTransformerDenoiser(feature_dim=144).to(device)
model = TemporalTransformerDenoiser(
    feature_dim=144,
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=256,
    dropout=0.1,
).to(device)

In [ ]:
model.load_state_dict(torch.load("pose_diffusion_transformer.pth"))  # Or your saved model path

In [ ]:
datamodule = PoseDataModule(finetune=None)

# IMPORTANT: must call setup
datamodule.setup(stage='fit')

# Get train loader
train_loader = datamodule.train_dataloader()
val_loader = datamodule.val_dataloader()

In [ ]:
datamodule = PoseDataModule(finetune=None)

# IMPORTANT: must call setup
datamodule.setup(stage='fit')

# Get train loader
train_loader = datamodule.train_dataloader()
val_loader = datamodule.val_dataloader()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
k = 0

for i, batch in enumerate(train_loader):
    if i == k:
        break

(inputs, input_lengths), (outputs, output_lengths) = batch

x = outputs["poses"].to(device)
trans = outputs["trans"].to(device)

B, T, Fdim = x.shape
lengths = output_lengths["poses"]

In [ ]:
gt_batch = x


In [ ]:

from articulate.math import r6d_to_rotation_matrix
from viewers.smpl_viewer import SMPLViewer
viewer = SMPLViewer(fps=25)

batch_size = B

for b in range(batch_size):
    # valid length for this sequence
    L = int(lengths[b])

    # Take only valid frames for this sequence
    gt_pose_r6d   = gt_batch[b, :L]      # (L, 144)
    tran_gt       = trans[b, :L]    # (L, 3)
    # Convert R6D to rotation matrices
    gt_pose_rot = r6d_to_rotation_matrix(
        gt_pose_r6d.view(-1, 24, 6)
    ).view(-1, 24, 3, 3)


    #print(f"Visualizing sequence {b+1}/{batch_size}, length = {L}")
    viewer.view(gt_pose_rot, tran_gt, gt_pose_rot, tran_gt, with_tran=True)

In [ ]:

from articulate.math import r6d_to_rotation_matrix
from viewers.smpl_viewer import SMPLViewer



model.eval()

k = 0

for i, batch in enumerate(val_loader):
    if i == k:
        break

(inputs, input_lengths), (outputs, output_lengths) = batch

x = outputs["poses"].to(device)
trans = outputs["trans"].to(device)

B, T, Fdim = x.shape
lengths = output_lengths["poses"]

# ---------- DDPM reverse (ancestral) sampling ----------
_betas    = betas.to(device)
_alphas   = alphas.to(device)
_alpha_bar = alpha_bar.to(device)

x_t = torch.randn_like(x)                         # start from pure noise

with torch.no_grad():
    for t_val in reversed(range(TIMESTEPS)):
        t_tensor = torch.full((B,), t_val, device=device, dtype=torch.long)

        noise_pred = model(x_t, t_tensor)          # predicted noise

        beta_t     = _betas[t_val]
        alpha_t    = _alphas[t_val]
        alpha_bar_t = _alpha_bar[t_val]

        # DDPM mean: mu_theta(x_t, t)
        coeff = beta_t / torch.sqrt(1.0 - alpha_bar_t)
        mean  = (1.0 / torch.sqrt(alpha_t)) * (x_t - coeff * noise_pred)

        if t_val > 0:
            sigma = torch.sqrt(beta_t)
            x_t = mean + sigma * torch.randn_like(x_t)
        else:
            x_t = mean                              # no noise at t = 0

pred = x_t                                          # final denoised prediction

gt_batch = x
pred_batch = pred
tran_batch = trans
lengths = output_lengths["poses"]

viewer = SMPLViewer(fps=25)


batch_size = 10

for b in range(batch_size):
    # valid length for this sequence
    L = int(lengths[b])

    # Take only valid frames for this sequence
    gt_pose_r6d   = gt_batch[b, :L]      # (L, 144)
    pred_pose_r6d = pred_batch[b, :L]    # (L, 144)
    tran_gt       = tran_batch[b, :L]    # (L, 3)
    tran_pred     = tran_gt              # model predicts pose, not translation

    # Convert R6D to rotation matrices
    gt_pose_rot = r6d_to_rotation_matrix(
        gt_pose_r6d.view(-1, 24, 6)
    ).view(-1, 24, 3, 3)

    pred_pose_rot = r6d_to_rotation_matrix(
        pred_pose_r6d.view(-1, 24, 6)
    ).view(-1, 24, 3, 3)

    #print(f"Visualizing sequence {b+1}/{batch_size}, length = {L}")
    viewer.view(pred_pose_rot, tran_pred, gt_pose_rot, tran_gt, with_tran=True)

# encoder

In [ ]:
import math
import numpy as np
import torch
torch.set_printoptions(sci_mode=False)
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
from typing import List
import random
import lightning as L
from tqdm import tqdm 

import articulate as art
from config import *
from utils import *
from helpers import *


import os
from articulate.math import r6d_to_rotation_matrix
from viewers.smpl_viewer import SMPLViewer
from articulate import model as art_model
from utils.model_utils import reduced_pose_to_full
from articulate.math import r6d_to_rotation_matrix
from articulate.evaluator import MeanPerJointErrorEvaluator
from articulate.math import RotationRepresentation


class PoseDataset(Dataset):
    def __init__(self, fold: str='train', evaluate: str=None, finetune: str=None):
        super().__init__()
        self.fold = fold
        self.evaluate = evaluate
        self.finetune = finetune
        self.bodymodel = art.model.ParametricModel(paths.smpl_file)
        self.combos = list(amass.combos.items())
        self.data = self._prepare_dataset()

    def _get_data_files(self, data_folder):
        if self.fold == 'train':
            return self._get_train_files(data_folder)
        elif self.fold == 'test':
            return self._get_test_files()
        else:
            raise ValueError(f"Unknown data fold: {self.fold}.")

    def _get_train_files(self, data_folder):
        if self.finetune:
            return [datasets.finetune_datasets[self.finetune]]
        else:
            return [x.name for x in data_folder.iterdir() if not x.is_dir()]

    def _get_test_files(self):
        return [datasets.test_datasets[self.evaluate]]

    def _prepare_dataset(self):
        data_folder = paths.processed_datasets / ('eval' if (self.finetune or self.evaluate) else '')
        data_files = self._get_data_files(data_folder)

        print(f"\n{'='*60}")
        print(f"Loading datasets for {self.fold.upper()} mode")
        print(f"Datasets: {data_files}")
        print(f"{'='*60}\n")

        data = {key: [] for key in [
            'imu_inputs', 'pose_outputs', 'joint_outputs',
            'tran_outputs', 'vel_outputs', 'foot_outputs'
        ]}

        for data_file in tqdm(data_files):
            try:
                file_data = torch.load(data_folder / data_file)
                self._process_file_data(file_data, data)
            except Exception as e:
                print(f"Error processing {data_file}: {e}")

        # print("\nTotal pose windows stored:", len(data['pose_outputs']))
        return data

    def _process_file_data(self, file_data, data):
        accs, oris, poses, trans = file_data['acc'], file_data['ori'], file_data['pose'], file_data['tran']
        joints = file_data.get('joint', [None] * len(poses))
        foots = file_data.get('contact', [None] * len(poses))

        for acc, ori, pose, tran, joint, foot in zip(accs, oris, poses, trans, joints, foots):


            acc, ori = acc[:, :5]/amass.acc_scale, ori[:, :5]

            pose_global, joint = self.bodymodel.forward_kinematics(
                pose=pose.view(-1, 216)
            )

            pose = pose if self.evaluate else pose_global.view(-1, 24, 3, 3)
            joint = joint.view(-1, 24, 3)

            # print("Pose after forward kinematics:", pose.shape)

            self._process_combo_data(acc, ori, pose, joint, tran, foot, data)

    def _process_combo_data(self, acc, ori, pose, joint, tran, foot, data):

        for combo_name, c in self.combos:


            combo_acc = torch.zeros_like(acc)
            combo_ori = torch.zeros_like(ori)
            combo_acc[:, c] = acc[:, c]
            combo_ori[:, c] = ori[:, c]

            imu_input = torch.cat([combo_acc.flatten(1), combo_ori.flatten(1)], dim=1)

            data_len = len(imu_input) if self.evaluate else datasets.window_length


            for key, value in zip(
                ['imu_inputs', 'pose_outputs', 'joint_outputs', 'tran_outputs'],
                [imu_input, pose, joint, tran]
            ):
                # data[key].extend(torch.split(value, data_len))
                splits = torch.split(value, data_len)

                # remove last if smaller than full window
                if splits[-1].shape[0] < data_len:
                    splits = splits[:-1]

                data[key].extend(splits)

            if not (self.evaluate or self.finetune):
                self._process_translation_data(joint, tran, foot, data_len, data)

    def _process_translation_data(self, joint, tran, foot, data_len, data):

        # print("\nProcessing translation module")

        root_vel = torch.cat((torch.zeros(1, 3), tran[1:] - tran[:-1]))
        vel = torch.cat((torch.zeros(1, 24, 3), torch.diff(joint, dim=0)))
        vel[:, 0] = root_vel

        vel = vel * (datasets.fps / amass.vel_scale)

        vel_splits = torch.split(vel, data_len)

        # print("Velocity windows created:", len(vel_splits))
        # print("Velocity window shape:", vel_splits[0].shape)

        data['vel_outputs'].extend(vel_splits)
        data['foot_outputs'].extend(torch.split(foot, data_len))

    def __getitem__(self, idx):

        imu = self.data['imu_inputs'][idx].float()
        joint = self.data['joint_outputs'][idx].float()
        tran = self.data['tran_outputs'][idx].float()


        num_pred_joints = len(amass.pred_joints_set)

        pose = art.math.rotation_matrix_to_r6d(
            self.data['pose_outputs'][idx]
        ).reshape(-1, num_pred_joints, 6)[:, amass.pred_joints_set] \
         .reshape(-1, 6*num_pred_joints)

        # print("Pose AFTER r6d conversion shape:", pose.shape)
        # print("#"*60)

        if self.evaluate or self.finetune:
            return imu, pose, joint, tran

        vel = self.data['vel_outputs'][idx].float()
        contact = self.data['foot_outputs'][idx].float()

        return imu, pose, joint, tran, vel, contact

    def __len__(self):
        return len(self.data['imu_inputs'])
    




def pad_seq(batch):
    """Pad sequences to same length for RNN."""
    def _pad(sequence):
        padded = nn.utils.rnn.pad_sequence(sequence, batch_first=True)
        lengths = [seq.shape[0] for seq in sequence]
        return padded, lengths

    inputs, poses, joints, trans = zip(*[(item[0], item[1], item[2], item[3]) for item in batch])
    inputs, input_lengths = _pad(inputs)
    poses, pose_lengths = _pad(poses)
    joints, joint_lengths = _pad(joints)
    trans, tran_lengths = _pad(trans)
    
    outputs = {'poses': poses, 'joints': joints, 'trans': trans}
    output_lengths = {'poses': pose_lengths, 'joints': joint_lengths, 'trans': tran_lengths}

    if len(batch[0]) > 5: # include velocity and foot contact, if available
        vels, foots = zip(*[(item[4], item[5]) for item in batch])

        # foot contact 
        foot_contacts, foot_contact_lengths = _pad(foots)
        outputs['foot_contacts'], output_lengths['foot_contacts'] = foot_contacts, foot_contact_lengths

        # root velocities
        vels, vel_lengths = _pad(vels)
        outputs['vels'], output_lengths['vels'] = vels, vel_lengths

    return (inputs, input_lengths), (outputs, output_lengths)


class PoseDataModule(L.LightningDataModule):
    def __init__(self, finetune: str = None):
        super().__init__()
        self.finetune = finetune
        self.hypers = finetune_hypers if self.finetune else train_hypers

    def setup(self, stage: str):
        if stage == 'fit':
            dataset = PoseDataset(fold='train', finetune=self.finetune)
            train_size = int(0.9 * len(dataset))
            val_size = len(dataset) - train_size
            self.train_dataset, self.val_dataset = random_split(dataset, [train_size, val_size])
        elif stage == 'test':
            self.test_dataset = PoseDataset(fold='test', finetune=self.finetune)

    def _dataloader(self, dataset):
        return DataLoader(
            dataset, 
            batch_size=self.hypers.batch_size, 
            collate_fn=pad_seq, 
            num_workers=0, #self.hypers.num_workers, 
            shuffle=True, 
            drop_last=True
        )

    def train_dataloader(self):
        return self._dataloader(self.train_dataset)

    def val_dataloader(self):
        return self._dataloader(self.val_dataset)

    def test_dataloader(self):
        return self._dataloader(self.test_dataset)


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Add positional encoding to input.

        x: (B, T, D)
        """
        T = x.size(1)
        return x + self.pe[:T].unsqueeze(0)


class TemporalTransformerDenoiser(nn.Module):
    def __init__(
        self,
        feature_dim: int,
        d_model: int = 256,
        nhead: int = 8,
        num_layers: int = 4,
        dim_feedforward: int = 512,
        dropout: float = 0.1,
    ):
        """Temporal Transformer denoiser.

        Inputs / outputs: (B, T, feature_dim)
        """
        super().__init__()

        self.in_proj = nn.Linear(feature_dim, d_model)
        self.pos_enc = SinusoidalPositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.out_proj = nn.Linear(d_model, feature_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, F)
        h = self.in_proj(x)
        h = self.pos_enc(h)
        h = self.encoder(h)  # (B, T, d_model)
        return self.out_proj(h)  # (B, T, F)
    
def validate(val_loader, model, device, mpj_eval):
    from utils.model_utils import reduced_pose_to_full
    from articulate.math import r6d_to_rotation_matrix
    import torch
    import torch.nn.functional as F

    model.eval()

    val_weighted_loss = 0.0
    val_mpj_sum = 0.0
    val_total_frames = 0

    with torch.no_grad():
        for batch in val_loader:

            (inputs, input_lengths), (outputs, output_lengths) = batch

            x = outputs["poses"].to(device)
            lengths = torch.as_tensor(output_lengths["poses"], device=device)

            B, T, F = x.shape

            # NO noise during validation
            pred = model(x)

            loss_matrix = nn.functional.mse_loss(pred, x, reduction="none")

            mask = torch.arange(T, device=device)[None, :] < lengths[:, None]
            mask = mask.unsqueeze(-1).float()

            masked_loss = (loss_matrix * mask).sum()
            num_valid = mask.sum()

            # compute MPJ error for valid frames
            if num_valid > 0:
                if F % 9 == 0:
                    in_rep = 9
                elif F % 6 == 0:
                    in_rep = 6
                else:
                    raise RuntimeError(f"Unknown pose feature dim F={F}")

                num_joints = F // in_rep
                pose_pred = pred.view(B * T, num_joints, in_rep)
                pose_true = x.view(B * T, num_joints, in_rep)
                valid_idx = mask.view(-1).bool()

                if valid_idx.any():
                    if in_rep == 6:
                        pose_pred_mat = r6d_to_rotation_matrix(pose_pred)
                        pose_true_mat = r6d_to_rotation_matrix(pose_true)
                    else:
                        pose_pred_mat = pose_pred.view(-1, num_joints, 3, 3)
                        pose_true_mat = pose_true.view(-1, num_joints, 3, 3)

                    pose_pred_mat = pose_pred_mat[valid_idx]
                    pose_true_mat = pose_true_mat[valid_idx]

                    # model joint count
                    try:
                        j, _ = mpj_eval.model.get_zero_pose_joint_and_vertex(None)
                        model_num_j = j.shape[1] if j.dim() == 3 else j.shape[0]
                    except Exception:
                        model_num_j = pose_pred_mat.shape[1]

                    # expand reduced -> full (fix for shape mismatch)
                    if pose_pred_mat.shape[1] != model_num_j:
                        N_valid = pose_pred_mat.shape[0]
                        pose_pred_full = reduced_pose_to_full(pose_pred_mat.unsqueeze(1)).view(N_valid, model_num_j, 3, 3)
                        pose_true_full = reduced_pose_to_full(pose_true_mat.unsqueeze(1)).view(N_valid, model_num_j, 3, 3)
                        pose_pred_mat = pose_pred_full
                        pose_true_mat = pose_true_full

                    mpj_tensor = mpj_eval(pose_pred_mat, pose_true_mat)
                    val_mpj_sum += mpj_tensor[0].item() * valid_idx.sum().item()

            val_weighted_loss += masked_loss.item()
            val_total_frames += num_valid.item()

    if val_total_frames > 0:
        val_loss = val_weighted_loss / val_total_frames
        val_mpj = val_mpj_sum / val_total_frames
    else:
        val_loss = 0.0
        val_mpj = 0.0

    print(f"Validation Loss: {val_loss:.6f} | MPJPE: {val_mpj:.6f}")

    return val_loss, val_mpj



def training(train_loader,val_loader, model, num_epochs=1, device=None, patience: int = 10, min_delta: float = 1e-4):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model = model.to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=3e-5,
        weight_decay=1e-4
    )

    criterion = nn.MSELoss(reduction="none")  # elementwise for masking

    best_epoch_loss = None
    epochs_no_improve = 0

    for epoch in range(num_epochs):

        model.train()
        epoch_weighted_loss = 0.0
        total_frames = 0

        for batch in train_loader:

            (inputs, input_lengths), (outputs, output_lengths) = batch

            # -------------------------------------------------
            # Get Pose Data
            # -------------------------------------------------
            x = outputs["poses"].to(device)
            lengths = torch.as_tensor(
                output_lengths["poses"],
                device=device
            )

            B, T, F = x.shape

            # -------------------------------------------------
            # Optional: Add Noise (Denoising Training)
            # -------------------------------------------------
            noise = torch.randn_like(x) * 0.01
            noisy_x = x + noise

            # -------------------------------------------------
            # Forward
            # -------------------------------------------------
            pred = model(noisy_x)

            loss_matrix = criterion(pred, x)

            # -------------------------------------------------
            # Mask Padding Frames
            # -------------------------------------------------
            mask = torch.arange(T, device=device)[None, :] < lengths[:, None]
            mask = mask.unsqueeze(-1).float()

            masked_loss = (loss_matrix * mask).sum()
            num_valid = mask.sum()

            if num_valid > 0:
                loss = masked_loss / num_valid
            else:
                loss = torch.tensor(
                    0.0,
                    device=device,
                    requires_grad=True
                )

            # -------------------------------------------------
            # Backprop
            # -------------------------------------------------
            optimizer.zero_grad()
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            # -------------------------------------------------
            # Logging
            # -------------------------------------------------
            with torch.no_grad():
                grad_norm = 0.0
                for p in model.parameters():
                    if p.grad is not None:
                        grad_norm += p.grad.data.norm(2).item() ** 2
                grad_norm = grad_norm ** 0.5

            print(
                f"Batch Loss: {loss.item():.6f} | "
                f"Grad Norm: {grad_norm:.4f}"
            )

            epoch_weighted_loss += masked_loss.item()
            total_frames += num_valid.item()

        # -------------------------------------------------
        # Epoch Summary
        # -------------------------------------------------
        if total_frames > 0:
            epoch_loss = epoch_weighted_loss / total_frames
        else:
            epoch_loss = 0.0

        print("===================================")
        print(f"Epoch {epoch}")
        print(f"Epoch Loss: {epoch_loss:.6f}")
        print("===================================")

        # -------------------------------------------------
        # Early Stopping
        # -------------------------------------------------
        if best_epoch_loss is None or epoch_loss < best_epoch_loss - min_delta:
            best_epoch_loss = epoch_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), "temporal_transformer_model_best.pth")
        else:
            epochs_no_improve += 1
            print(f"No improvement for {epochs_no_improve}/{patience} epochs.")
            if epochs_no_improve >= patience:
                print("Early stopping triggered.")
                break

        mpj_eval = MeanPerJointErrorEvaluator(
        official_model_file=paths.smpl_file,
        rep=RotationRepresentation.ROTATION_MATRIX,
        device=device
            )

        val_loss, val_mpj = validate(
            val_loader=val_loader,
            model=model,
            device=device,
            mpj_eval=mpj_eval
        )
        print(f"Validation Loss: {val_loss:.6f} | MPJPE: {val_mpj:.6f}")
    torch.save(model.state_dict(), f"temporal_transformer_model.pth")




In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"


datamodule = PoseDataModule(finetune=None)

# IMPORTANT: must call setup
datamodule.setup(stage='fit')

# Get train loader
train_loader = datamodule.train_dataloader()
val_loader = datamodule.val_dataloader()


# model = TemporalDenoiseUNet(feature_dim=144).to(device)
model = TemporalTransformerDenoiser(
    feature_dim=144,
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=256,
    dropout=0.1,
).to(device)

In [ ]:
model.load_state_dict(torch.load("temporal_transformer_model_best.pth"))  # Load best model from training

In [ ]:
# mpj_eval = MeanPerJointErrorEvaluator(
# official_model_file=paths.smpl_file,
# rep=RotationRepresentation.ROTATION_MATRIX,
# device=device
#     )

# val_loss, val_mpj = validate(
#     val_loader=val_loader,
#     model=model,
#     device=device,
#     mpj_eval=mpj_eval
# )
# print(f"Validation Loss: {val_loss:.6f} | MPJPE: {val_mpj:.6f}")

In [ ]:
from articulate.math import r6d_to_rotation_matrix
from viewers.smpl_viewer import SMPLViewer
import os
os.environ["GT"] = "1"
model.eval()

# 180° rotation around X-axis to flip the body upright
flip_rot = torch.eye(3, device=device)
flip_rot[1, 1] = -1
flip_rot[2, 2] = -1

k = 0

for i, batch in enumerate(val_loader):
    if i == k:
        break

(inputs, input_lengths), (outputs, output_lengths) = batch

x = outputs["poses"].to(device)
trans = outputs["trans"].to(device)

B, T, Fdim = x.shape

noise = torch.randn_like(x) * 0.01
noisy_x = x + noise

with torch.no_grad():
    pred = model(noisy_x)

gt_batch = x
pred_batch = pred
tran_batch = trans
lengths = output_lengths["poses"]

viewer = SMPLViewer(fps=25)
bodymodel = viewer.bodymodel

batch_size = 2

for b in range(batch_size):
    L = int(lengths[b])

    gt_pose_r6d   = gt_batch[b, :L]
    pred_pose_r6d = pred_batch[b, :L]
    tran_gt       = tran_batch[b, :L]
    tran_pred     = tran_gt

    # Convert R6D -> global rotation matrices
    gt_pose_rot = r6d_to_rotation_matrix(
        gt_pose_r6d.view(-1, 24, 6)
    ).view(-1, 24, 3, 3)

    pred_pose_rot = r6d_to_rotation_matrix(
        pred_pose_r6d.view(-1, 24, 6)
    ).view(-1, 24, 3, 3)

    # Convert global -> local (viewer applies FK internally)
    gt_pose_local = bodymodel.inverse_kinematics_R(gt_pose_rot)
    pred_pose_local = bodymodel.inverse_kinematics_R(pred_pose_rot)

    # Flip root joint to stand upright
    gt_pose_local[:, 0] = flip_rot @ gt_pose_local[:, 0]
    pred_pose_local[:, 0] = flip_rot @ pred_pose_local[:, 0]

    viewer.view(pred_pose_local, tran_pred, gt_pose_local, tran_gt, with_tran=False)


In [ ]:

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Add positional encoding to input.

        x: (B, T, D)
        """
        T = x.size(1)
        return x + self.pe[:T].unsqueeze(0)


class TemporalTransformerDenoiser(nn.Module):
    def __init__(
        self,
        feature_dim: int,
        d_model: int = 256,
        nhead: int = 8,
        num_layers: int = 4,
        dim_feedforward: int = 512,
        dropout: float = 0.1,
    ):
        """Temporal Transformer denoiser.

        Inputs / outputs: (B, T, feature_dim)
        """
        super().__init__()

        self.in_proj = nn.Linear(feature_dim, d_model)
        self.pos_enc = SinusoidalPositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.out_proj = nn.Linear(d_model, feature_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, F)
        h = self.in_proj(x)
        h = self.pos_enc(h)
        h = self.encoder(h)  # (B, T, d_model)
        return self.out_proj(h)  # (B, T, F)

# diffusion model

In [10]:


import math
import numpy as np
import torch
torch.set_printoptions(sci_mode=False)
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
from typing import List
import random
import lightning as L
from tqdm import tqdm 

import articulate as art
from config import *
from utils import *
from helpers import *


import os
from articulate.math import r6d_to_rotation_matrix
from viewers.smpl_viewer import SMPLViewer
from articulate import model as art_model
from utils.model_utils import reduced_pose_to_full
from articulate.math import r6d_to_rotation_matrix
from articulate.evaluator import MeanPerJointErrorEvaluator
from articulate.math import RotationRepresentation


class PoseDataset(Dataset):
    def __init__(self, fold: str='train', evaluate: str=None, finetune: str=None):
        super().__init__()
        self.fold = fold
        self.evaluate = evaluate
        self.finetune = finetune
        self.bodymodel = art.model.ParametricModel(paths.smpl_file)
        self.combos = list(amass.combos.items())
        self.data = self._prepare_dataset()

    def _get_data_files(self, data_folder):
        if self.fold == 'train':
            return self._get_train_files(data_folder)
        elif self.fold == 'test':
            return self._get_test_files()
        else:
            raise ValueError(f"Unknown data fold: {self.fold}.")

    def _get_train_files(self, data_folder):
        if self.finetune:
            return [datasets.finetune_datasets[self.finetune]]
        else:
            return [x.name for x in data_folder.iterdir() if not x.is_dir()]

    def _get_test_files(self):
        return [datasets.test_datasets[self.evaluate]]

    def _prepare_dataset(self):
        data_folder = paths.processed_datasets / ('eval' if (self.finetune or self.evaluate) else '')
        data_files = self._get_data_files(data_folder)

        print(f"\n{'='*60}")
        print(f"Loading datasets for {self.fold.upper()} mode")
        print(f"Datasets: {data_files}")
        print(f"{'='*60}\n")

        data = {key: [] for key in [
            'imu_inputs', 'pose_outputs', 'joint_outputs',
            'tran_outputs', 'vel_outputs', 'foot_outputs'
        ]}

        for data_file in tqdm(data_files):
            try:
                file_data = torch.load(data_folder / data_file)
                self._process_file_data(file_data, data)
            except Exception as e:
                print(f"Error processing {data_file}: {e}")


        return data

    def _process_file_data(self, file_data, data):
        accs, oris, poses, trans = file_data['acc'], file_data['ori'], file_data['pose'], file_data['tran']
        joints = file_data.get('joint', [None] * len(poses))
        foots = file_data.get('contact', [None] * len(poses))

        for acc, ori, pose, tran, joint, foot in zip(accs, oris, poses, trans, joints, foots):

            acc, ori = acc[:, :5]/amass.acc_scale, ori[:, :5]

            pose_global, joint = self.bodymodel.forward_kinematics(
                pose=pose.view(-1, 216)
            )

            pose = pose if self.evaluate else pose_global.view(-1, 24, 3, 3)
            joint = joint.view(-1, 24, 3)


            self._process_combo_data(acc, ori, pose, joint, tran, foot, data)

    def _process_combo_data(self, acc, ori, pose, joint, tran, foot, data):

        for combo_name, c in self.combos:


            combo_acc = torch.zeros_like(acc)
            combo_ori = torch.zeros_like(ori)
            combo_acc[:, c] = acc[:, c]
            combo_ori[:, c] = ori[:, c]

            imu_input = torch.cat([combo_acc.flatten(1), combo_ori.flatten(1)], dim=1)

            data_len = len(imu_input) if self.evaluate else datasets.window_length


            for key, value in zip(
                ['imu_inputs', 'pose_outputs', 'joint_outputs', 'tran_outputs'],
                [imu_input, pose, joint, tran]
            ):
                # data[key].extend(torch.split(value, data_len))
                splits = torch.split(value, data_len)

                # remove last if smaller than full window
                if splits[-1].shape[0] < data_len:
                    splits = splits[:-1]

                data[key].extend(splits)

            if not (self.evaluate or self.finetune):
                self._process_translation_data(joint, tran, foot, data_len, data)

    def _process_translation_data(self, joint, tran, foot, data_len, data):

        root_vel = torch.cat((torch.zeros(1, 3), tran[1:] - tran[:-1]))
        vel = torch.cat((torch.zeros(1, 24, 3), torch.diff(joint, dim=0)))
        vel[:, 0] = root_vel

        vel = vel * (datasets.fps / amass.vel_scale)

        vel_splits = torch.split(vel, data_len)


        data['vel_outputs'].extend(vel_splits)
        data['foot_outputs'].extend(torch.split(foot, data_len))

    def __getitem__(self, idx):

        imu = self.data['imu_inputs'][idx].float()
        joint = self.data['joint_outputs'][idx].float()
        tran = self.data['tran_outputs'][idx].float()

        num_pred_joints = len(amass.pred_joints_set)

        pose = art.math.rotation_matrix_to_r6d(
            self.data['pose_outputs'][idx]
        ).reshape(-1, num_pred_joints, 6)[:, amass.pred_joints_set] \
         .reshape(-1, 6*num_pred_joints)

        # print("Pose AFTER r6d conversion shape:", pose.shape)
        # print("#"*60)

        if self.evaluate or self.finetune:
            return imu, pose, joint, tran

        vel = self.data['vel_outputs'][idx].float()
        contact = self.data['foot_outputs'][idx].float()

        return imu, pose, joint, tran, vel, contact

    def __len__(self):
        return len(self.data['imu_inputs'])
    




def pad_seq(batch):
    """Pad sequences to same length for RNN."""
    def _pad(sequence):
        padded = nn.utils.rnn.pad_sequence(sequence, batch_first=True)
        lengths = [seq.shape[0] for seq in sequence]
        return padded, lengths

    inputs, poses, joints, trans = zip(*[(item[0], item[1], item[2], item[3]) for item in batch])
    inputs, input_lengths = _pad(inputs)
    poses, pose_lengths = _pad(poses)
    joints, joint_lengths = _pad(joints)
    trans, tran_lengths = _pad(trans)
    
    outputs = {'poses': poses, 'joints': joints, 'trans': trans}
    output_lengths = {'poses': pose_lengths, 'joints': joint_lengths, 'trans': tran_lengths}

    if len(batch[0]) > 5: # include velocity and foot contact, if available
        vels, foots = zip(*[(item[4], item[5]) for item in batch])

        # foot contact 
        foot_contacts, foot_contact_lengths = _pad(foots)
        outputs['foot_contacts'], output_lengths['foot_contacts'] = foot_contacts, foot_contact_lengths

        # root velocities
        vels, vel_lengths = _pad(vels)
        outputs['vels'], output_lengths['vels'] = vels, vel_lengths

    return (inputs, input_lengths), (outputs, output_lengths)


class PoseDataModule(L.LightningDataModule):
    def __init__(self, finetune: str = None):
        super().__init__()
        self.finetune = finetune
        self.hypers = finetune_hypers if self.finetune else train_hypers

    def setup(self, stage: str):
        if stage == 'fit':
            dataset = PoseDataset(fold='train', finetune=self.finetune)
            train_size = int(0.9 * len(dataset))
            val_size = len(dataset) - train_size
            self.train_dataset, self.val_dataset = random_split(dataset, [train_size, val_size])
        elif stage == 'test':
            self.test_dataset = PoseDataset(fold='test', finetune=self.finetune)

    def _dataloader(self, dataset):
        return DataLoader(
            dataset, 
            batch_size=self.hypers.batch_size, 
            collate_fn=pad_seq, 
            num_workers=0, #self.hypers.num_workers, 
            shuffle=True, 
            drop_last=True
        )

    def train_dataloader(self):
        return self._dataloader(self.train_dataset)

    def val_dataloader(self):
        return self._dataloader(self.val_dataset)

    def test_dataloader(self):
        return self._dataloader(self.test_dataset)


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Add positional encoding to input.

        x: (B, T, D)
        """
        T = x.size(1)
        return x + self.pe[:T].unsqueeze(0)


class SinusoidalTimestepEmbedding(nn.Module):
    """Sinusoidal embedding for diffusion timestep, projected through an MLP."""
    def __init__(self, d_model: int):
        super().__init__()
        self.d_model = d_model
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.SiLU(),
            nn.Linear(d_model * 4, d_model),
        )

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """t: (B,) integer timesteps -> (B, d_model)"""
        half = self.d_model // 2
        freqs = torch.exp(
            -math.log(10000.0) * torch.arange(half, device=t.device, dtype=torch.float32) / half
        )
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        if self.d_model % 2 == 1:
            emb = torch.cat([emb, torch.zeros_like(emb[:, :1])], dim=-1)
        return self.mlp(emb)


# ---------------------------------------------------------------------------
# Diffusion noise schedule helpers
# ---------------------------------------------------------------------------
def linear_beta_schedule(num_timesteps: int, beta_start: float = 1e-4, beta_end: float = 0.02):
    return torch.linspace(beta_start, beta_end, num_timesteps)


def cosine_beta_schedule(num_timesteps: int, s: float = 0.008):
    steps = torch.arange(num_timesteps + 1, dtype=torch.float64)
    alpha_bar = torch.cos(((steps / num_timesteps) + s) / (1 + s) * (math.pi / 2)) ** 2
    alpha_bar = alpha_bar / alpha_bar[0]
    betas = 1 - (alpha_bar[1:] / alpha_bar[:-1])
    return torch.clamp(betas, 0.0001, 0.9999).float()


class GaussianDiffusion(nn.Module):
    """Manages the forward (noising) and reverse (denoising) diffusion process."""
    def __init__(self, num_timesteps: int = 1000, schedule: str = "cosine"):
        super().__init__()
        self.num_timesteps = num_timesteps

        if schedule == "cosine":
            betas = cosine_beta_schedule(num_timesteps)
        else:
            betas = linear_beta_schedule(num_timesteps)

        alphas = 1.0 - betas
        alpha_bar = torch.cumprod(alphas, dim=0)
        alpha_bar_prev = torch.cat([torch.tensor([1.0]), alpha_bar[:-1]])

        # Register all as buffers so they move with .to(device)
        self.register_buffer("betas", betas)
        self.register_buffer("alphas", alphas)
        self.register_buffer("alpha_bar", alpha_bar)
        self.register_buffer("alpha_bar_prev", alpha_bar_prev)
        self.register_buffer("sqrt_alpha_bar", torch.sqrt(alpha_bar))
        self.register_buffer("sqrt_one_minus_alpha_bar", torch.sqrt(1.0 - alpha_bar))
        self.register_buffer("sqrt_recip_alpha", torch.sqrt(1.0 / alphas))
        self.register_buffer(
            "posterior_variance",
            betas * (1.0 - alpha_bar_prev) / (1.0 - alpha_bar),
        )

    def q_sample(self, x0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor = None):
        """Forward diffusion: q(x_t | x_0)."""
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_ab = self.sqrt_alpha_bar[t]
        sqrt_omab = self.sqrt_one_minus_alpha_bar[t]
        # broadcast to (B, 1, 1) if x0 is (B, T, D)
        while sqrt_ab.dim() < x0.dim():
            sqrt_ab = sqrt_ab.unsqueeze(-1)
            sqrt_omab = sqrt_omab.unsqueeze(-1)
        return sqrt_ab * x0 + sqrt_omab * noise, noise

    @torch.no_grad()
    def p_sample(self, model, x_t, t_index: int):
        """Single reverse step: p(x_{t-1} | x_t)."""
        B = x_t.shape[0]
        t_tensor = torch.full((B,), t_index, device=x_t.device, dtype=torch.long)
        noise_pred = model(x_t, t_tensor)
        beta = self.betas[t_index]
        sqrt_recip_alpha = self.sqrt_recip_alpha[t_index]
        sqrt_omab = self.sqrt_one_minus_alpha_bar[t_index]
        mean = sqrt_recip_alpha * (x_t - beta / sqrt_omab * noise_pred)
        if t_index > 0:
            sigma = torch.sqrt(self.posterior_variance[t_index])
            mean = mean + sigma * torch.randn_like(x_t)
        return mean

    @torch.no_grad()
    def sample(self, model, shape, device):
        """Full reverse sampling loop: x_T -> x_0."""
        x = torch.randn(shape, device=device)
        for t in reversed(range(self.num_timesteps)):
            x = self.p_sample(model, x, t)
        return x


class TemporalTransformerDiffusion(nn.Module):
    """Unconditional diffusion model (noise predictor) for pose + translation.

    Takes noisy (pose, tran) concatenated along feature dim + a diffusion
    timestep, and predicts the noise that was added.
    """
    def __init__(
        self,
        pose_dim: int,
        tran_dim: int = 3,
        d_model: int = 256,
        nhead: int = 8,
        num_layers: int = 4,
        dim_feedforward: int = 512,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.pose_dim = pose_dim
        self.tran_dim = tran_dim
        combined_dim = pose_dim + tran_dim

        self.in_proj = nn.Linear(combined_dim, d_model)
        self.time_emb = SinusoidalTimestepEmbedding(d_model)
        self.pos_enc = SinusoidalPositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # separate output heads predicting noise for pose and translation
        self.pose_head = nn.Linear(d_model, pose_dim)
        self.tran_head = nn.Linear(d_model, tran_dim)

    def forward(self, x_t: torch.Tensor, t: torch.Tensor):
        """Predict noise from noisy input + timestep.

        x_t : (B, T, pose_dim + tran_dim)  noisy concatenated pose+tran
        t   : (B,) integer diffusion timesteps
        Returns: (B, T, pose_dim + tran_dim) predicted noise
        """
        h = self.in_proj(x_t)                       # (B, T, d_model)
        t_emb = self.time_emb(t).unsqueeze(1)        # (B, 1, d_model)
        h = h + t_emb                                # broadcast add timestep info
        h = self.pos_enc(h)
        h = self.encoder(h)                          # (B, T, d_model)
        pose_noise = self.pose_head(h)
        tran_noise = self.tran_head(h)
        return torch.cat([pose_noise, tran_noise], dim=-1)
    
def validate(val_loader, model, diffusion, device):
    model.eval()
    val_loss_sum = 0.0
    val_total_frames = 0

    with torch.no_grad():
        for batch in val_loader:
            (inputs, input_lengths), (outputs, output_lengths) = batch

            pose = outputs["poses"].to(device)
            tran = outputs["trans"].to(device)
            lengths = torch.as_tensor(output_lengths["poses"], device=device)

            B, T, F_pose = pose.shape
            x0 = torch.cat([pose, tran], dim=-1)  # (B, T, pose_dim + tran_dim)

            # sample random timesteps
            t = torch.randint(0, diffusion.num_timesteps, (B,), device=device)
            noise = torch.randn_like(x0)
            x_t, _ = diffusion.q_sample(x0, t, noise)

            noise_pred = model(x_t, t)

            loss_matrix = nn.functional.mse_loss(noise_pred, noise, reduction="none")

            mask = torch.arange(T, device=device)[None, :] < lengths[:, None]
            frame_mask = mask.unsqueeze(-1).float()

            masked_loss = (loss_matrix * frame_mask).sum()
            num_valid = frame_mask.sum()

            val_loss_sum += masked_loss.item()
            val_total_frames += num_valid.item()

    val_loss = val_loss_sum / val_total_frames if val_total_frames > 0 else 0.0
    print(f"Validation Loss: {val_loss:.6f}")
    return val_loss


In [11]:
os.environ["GT"] = "1"
device = "cuda" if torch.cuda.is_available() else "cpu"

POSE_DIM = 144
TRAN_DIM = 3
NUM_DIFFUSION_STEPS = 1000

# --- load diffusion schedule + model ---
diffusion = GaussianDiffusion(
    num_timesteps=NUM_DIFFUSION_STEPS, schedule="cosine"
).to(device)

model = TemporalTransformerDiffusion(
    pose_dim=POSE_DIM, tran_dim=TRAN_DIM,
    d_model=128, nhead=4, num_layers=2,
    dim_feedforward=256, dropout=0.1,
).to(device)

In [12]:
checkpoint_path = "diffusion_model_best.pth"

In [13]:
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()

C:\Users\degu03\AppData\Local\Temp\ipykernel_16564\2852279867.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(checkpoint_path, map_locat

TemporalTransformerDiffusion(
  (in_proj): Linear(in_features=147, out_features=128, bias=True)
  (time_emb): SinusoidalTimestepEmbedding(
    (mlp): Sequential(
      (0): Linear(in_features=128, out_features=512, bias=True)
      (1): SiLU()
      (2): Linear(in_features=512, out_features=128, bias=True)
    )
  )
  (pos_enc): SinusoidalPositionalEncoding()
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
 

In [14]:
datamodule = PoseDataModule(finetune=None)
datamodule.setup(stage="fit")
val_loader = datamodule.val_dataloader()


Loading datasets for TRAIN mode
Datasets: ['ACCAD.pt', 'BioMotionLab_NTroje.pt', 'BMLhandball.pt', 'BMLmovi.pt', 'CMU.pt', 'DanceDB.pt', 'DFaust_67.pt', 'dip_test.pt', 'dip_train.pt', 'Eyes_Japan_Dataset.pt', 'HUMAN4D.pt', 'HumanEva.pt', 'MPI_HDM05.pt', 'MPI_Limits.pt', 'MPI_mosh.pt', 'SFU.pt', 'SSM_synced.pt', 'TCD_handMocap.pt', 'TotalCapture.pt', 'Transitions_mocap.pt']



  0%|          | 0/20 [00:00<?, ?it/s]C:\Users\degu03\AppData\Local\Temp\ipykernel_16564\2782543517.py:71: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  file_data = torch.lo

In [15]:
bodymodel = art.model.ParametricModel(paths.smpl_file, device=device)
mpjpe_evaluator = MeanPerJointErrorEvaluator(
    official_model_file=str(paths.smpl_file),
    rep=RotationRepresentation.ROTATION_MATRIX,
    device=device,
)

In [ ]:
# def validate_diffusion(val_loader, model, diffusion, device, mpjpe_evaluator, bodymodel, noise_level=50):
#     """Validate diffusion model: noise GT to timestep `noise_level`, denoise back, compute MPJPE."""
#     from articulate.math import r6d_to_rotation_matrix
#     import numpy as np

#     model.eval()

#     all_mpjpe = []
#     all_local_err = []
#     all_global_err = []
#     val_loss_sum = 0.0
#     val_total_frames = 0

#     POSE_DIM = 144

#     with torch.no_grad():
#         for batch in tqdm(val_loader, desc="Validating"):
#             (inputs, input_lengths), (outputs, output_lengths) = batch

#             gt_pose_r6d = outputs["poses"].to(device)   # (B, T, 144)
#             gt_tran     = outputs["trans"].to(device)    # (B, T, 3)
#             lengths     = output_lengths["poses"]

#             B, T, _ = gt_pose_r6d.shape

#             # concatenate pose + tran as x0
#             x0 = torch.cat([gt_pose_r6d, gt_tran], dim=-1)  # (B, T, 147)

#             # forward-noise to timestep `noise_level`
#             t = torch.full((B,), noise_level, device=device, dtype=torch.long)
#             noise = torch.randn_like(x0)
#             x_noisy, _ = diffusion.q_sample(x0, t, noise)

#             # reverse denoise from noise_level back to 0
#             x_denoised = x_noisy
#             for step in reversed(range(noise_level)):
#                 x_denoised = diffusion.p_sample(model, x_denoised, step)

#             pred_pose_r6d = x_denoised[..., :POSE_DIM]
#             pred_tran     = x_denoised[..., POSE_DIM:]

#             # MSE loss (reconstruction error)
#             lengths_t = torch.as_tensor(lengths, device=device)
#             mask = torch.arange(T, device=device)[None, :] < lengths_t[:, None]
#             frame_mask = mask.unsqueeze(-1).float()
#             loss_matrix = nn.functional.mse_loss(
#                 torch.cat([pred_pose_r6d, pred_tran], dim=-1), x0, reduction="none"
#             )
#             val_loss_sum += (loss_matrix * frame_mask).sum().item()
#             val_total_frames += frame_mask.sum().item()

#             # MPJPE per sample
#             for b in range(B):
#                 L = int(lengths[b])
#                 gt_r6d   = gt_pose_r6d[b, :L]
#                 pred_r6d = pred_pose_r6d[b, :L]

#                 gt_rot   = r6d_to_rotation_matrix(gt_r6d.reshape(-1, 24, 6)).reshape(-1, 24, 3, 3)
#                 pred_rot = r6d_to_rotation_matrix(pred_r6d.reshape(-1, 24, 6)).reshape(-1, 24, 3, 3)

#                 gt_local   = bodymodel.inverse_kinematics_R(gt_rot)
#                 pred_local = bodymodel.inverse_kinematics_R(pred_rot)

#                 error = mpjpe_evaluator(pred_local.view(L, -1), gt_local.view(L, -1))
#                 all_mpjpe.append(error[0].item() * 100)       # m -> cm
#                 all_local_err.append(error[1].item())          # degrees
#                 all_global_err.append(error[2].item())         # degrees

#     val_loss = val_loss_sum / val_total_frames if val_total_frames > 0 else 0.0

#     print(f"\n{'='*60}")
#     print(f"Noise level (t)   : {noise_level}")
#     print(f"Samples evaluated : {len(all_mpjpe)}")
#     print(f"Recon MSE Loss    : {val_loss:.6f}")
#     print(f"Mean MPJPE        : {np.mean(all_mpjpe):.2f} cm")
#     print(f"Mean Local Angle  : {np.mean(all_local_err):.2f}°")
#     print(f"Mean Global Angle : {np.mean(all_global_err):.2f}°")
#     print(f"{'='*60}")

#     return val_loss, np.mean(all_mpjpe), np.mean(all_local_err), np.mean(all_global_err)


# # --- Run it ---
# val_loss, mpjpe, local_err, global_err = validate_diffusion(
#     val_loader, model, diffusion, device, mpjpe_evaluator, bodymodel,
#     noise_level=50   # try 50, 100, 200, etc.
# )

In [8]:
# from articulate.math import r6d_to_rotation_matrix
# from viewers.smpl_viewer import SMPLViewer
# import os
# os.environ["GT"] = "1"
# model.eval()

# # 180° rotation around X-axis to flip the body upright
# flip_rot = torch.eye(3, device=device)
# flip_rot[1, 1] = -1
# flip_rot[2, 2] = -1

# NOISE_LEVEL = 50  # timestep to noise to, then denoise back

# k = 1  # skip first k batches (set 0 to use the very first batch)

# for i, batch in enumerate(val_loader):
#     if i == k:
#         break

# (inputs, input_lengths), (outputs, output_lengths) = batch

# gt_pose_r6d = outputs["poses"].to(device)   # (B, T, 144)
# gt_tran     = outputs["trans"].to(device)    # (B, T, 3)
# lengths     = output_lengths["poses"]

# B, T, _ = gt_pose_r6d.shape

# # Concatenate pose + tran as x0 (the clean signal)
# x0 = torch.cat([gt_pose_r6d, gt_tran], dim=-1)  # (B, T, 147)

# # Forward-noise to timestep NOISE_LEVEL
# t = torch.full((B,), NOISE_LEVEL, device=device, dtype=torch.long)
# noise = torch.randn_like(x0)
# x_noisy, _ = diffusion.q_sample(x0, t, noise)

# # Reverse denoise from NOISE_LEVEL back to 0
# with torch.no_grad():
#     x_denoised = x_noisy
#     for step in reversed(range(NOISE_LEVEL)):
#         x_denoised = diffusion.p_sample(model, x_denoised, step)

# # Split back into pose and translation
# pred_pose_r6d = x_denoised[..., :POSE_DIM]   # (B, T, 144)
# pred_tran     = x_denoised[..., POSE_DIM:]    # (B, T, 3)

# viewer = SMPLViewer(fps=25)
# bodymodel_vis = viewer.bodymodel

# batch_size = 2

# for b in range(batch_size):
#     L = int(lengths[b])

#     gt_r6d   = gt_pose_r6d[b, :L]
#     pred_r6d = pred_pose_r6d[b, :L]
#     tran_gt  = gt_tran[b, :L]
#     tran_pred = pred_tran[b, :L]

#     # Convert R6D -> global rotation matrices (T, 24, 3, 3)
#     gt_rot   = r6d_to_rotation_matrix(gt_r6d.view(-1, 24, 6)).view(-1, 24, 3, 3)
#     pred_rot = r6d_to_rotation_matrix(pred_r6d.view(-1, 24, 6)).view(-1, 24, 3, 3)

#     # Convert global -> local (viewer applies FK internally)
#     gt_local   = bodymodel_vis.inverse_kinematics_R(gt_rot)
#     pred_local = bodymodel_vis.inverse_kinematics_R(pred_rot)

#     # Flip root joint to stand upright
#     gt_local[:, 0]   = flip_rot @ gt_local[:, 0]
#     pred_local[:, 0] = flip_rot @ pred_local[:, 0]

#     viewer.view(pred_local, tran_pred, gt_local, tran_gt, with_tran=True)

100%|##########| 125/125 [00:02<00:00, 59.67it/s]


In [16]:
import numpy as np
model.eval()

NOISE_LEVELS = [80, 100]
results = []

for NOISE_LEVEL in NOISE_LEVELS:
    all_mpjpe = []
    all_local_err = []
    all_global_err = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Evaluating MPJPE (noise={NOISE_LEVEL})"):
            (inputs, input_lengths), (outputs, output_lengths) = batch

            gt_pose_r6d = outputs["poses"].to(device)
            gt_tran     = outputs["trans"].to(device)
            lengths     = output_lengths["poses"]

            B, T, _ = gt_pose_r6d.shape

            x0 = torch.cat([gt_pose_r6d, gt_tran], dim=-1)

            t = torch.full((B,), NOISE_LEVEL, device=device, dtype=torch.long)
            noise = torch.randn_like(x0)
            x_noisy, _ = diffusion.q_sample(x0, t, noise)

            x_denoised = x_noisy
            for step in reversed(range(NOISE_LEVEL)):
                x_denoised = diffusion.p_sample(model, x_denoised, step)

            pred_pose_r6d = x_denoised[..., :POSE_DIM]
            pred_tran     = x_denoised[..., POSE_DIM:]

            for b in range(B):
                L = int(lengths[b])
                gt_r6d   = gt_pose_r6d[b, :L]
                pred_r6d = pred_pose_r6d[b, :L]

                gt_rot   = r6d_to_rotation_matrix(gt_r6d.view(-1, 24, 6)).view(-1, 24, 3, 3)
                pred_rot = r6d_to_rotation_matrix(pred_r6d.view(-1, 24, 6)).view(-1, 24, 3, 3)

                gt_local   = bodymodel.inverse_kinematics_R(gt_rot)
                pred_local = bodymodel.inverse_kinematics_R(pred_rot)

                error = mpjpe_evaluator(pred_local.view(L, -1), gt_local.view(L, -1))
                all_mpjpe.append(error[0].item() * 100)
                all_local_err.append(error[1].item())
                all_global_err.append(error[2].item())

    mean_mpjpe = np.mean(all_mpjpe)
    mean_local = np.mean(all_local_err)
    mean_global = np.mean(all_global_err)

    results.append({
        "noise_level": NOISE_LEVEL,
        "samples": len(all_mpjpe),
        "mpjpe": mean_mpjpe,
        "local_angle": mean_local,
        "global_angle": mean_global,
    })

    print(f"\n{'='*60}")
    print(f"Noise level (t)   : {NOISE_LEVEL}")
    print(f"Samples evaluated : {len(all_mpjpe)}")
    print(f"Mean MPJPE        : {mean_mpjpe:.2f} cm")
    print(f"Mean Local Angle  : {mean_local:.2f}\u00b0")
    print(f"Mean Global Angle : {mean_global:.2f}\u00b0")
    print(f"{'='*60}")

# Save results to a text file
output_path = "mpjpe_noise_level_results.txt"
with open(output_path, "w") as f:
    for r in results:
        f.write(f"{'='*60}\n")
        f.write(f"Noise level (t)   : {r['noise_level']}\n")
        f.write(f"Samples evaluated : {r['samples']}\n")
        f.write(f"Mean MPJPE        : {r['mpjpe']:.2f} cm\n")
        f.write(f"Mean Local Angle  : {r['local_angle']:.2f}\u00b0\n")
        f.write(f"Mean Global Angle : {r['global_angle']:.2f}\u00b0\n")
    f.write(f"{'='*60}\n")

print(f"\nResults saved to {output_path}")

Evaluating MPJPE (noise=80): 100%|██████████| 130/130 [19:53<00:00,  9.18s/it]



Noise level (t)   : 80
Samples evaluated : 33280
Mean MPJPE        : 4.13 cm
Mean Local Angle  : 7.81°
Mean Global Angle : 8.59°


Evaluating MPJPE (noise=100): 100%|██████████| 130/130 [19:42<00:00,  9.10s/it]


Noise level (t)   : 100
Samples evaluated : 33280
Mean MPJPE        : 4.82 cm
Mean Local Angle  : 8.70°
Mean Global Angle : 10.11°

Results saved to mpjpe_noise_level_results.txt
